# Task B — Training-Time Necessity (Kaggle runner)

Runs on Kaggle because 27 runs x 100,000 env steps (~55 CPU-min/run measured locally on this project's MacBook Air; ~25h serial) is too large for the project's usual local iteration loop.

**Before running:** Add the Kaggle Dataset built from the `kaggle/taskB/` folder of the project repo as an **Input** to this notebook (`+ Add Input` -> Datasets -> upload that folder as a new dataset). It contains `src/` (this project's own world model / RSSM / constrained-hook code, unmodified from the local run except the hook module, which is new) and `taskB_program.py` (the full implementation, a byte-identical port of the local `run_taskB_training_necessity.py`'s core training loop). Rename `INPUT_ROOT` below to match your dataset's slug if different.

**Resumability:** every (task, condition, seed) cell's result is written to `/kaggle/working/taskB_results/<task>_<condition>_<seed>/cell_result.json` (plus `model.pt`, `states.npz`, `loss_history.json`, `v_history.json`) the moment it finishes. Re-running this notebook skips every cell already on disk.

**Pilot first (§3.6 of the spec):** cartpole seed 3101, all 3 conditions, before committing to the full 27 (or 9, under the declared fallback).

In [ ]:
import sys, os

INPUT_ROOT = '/kaggle/input/taskb-signal-check-inputs'  # <-- rename to your dataset's slug
assert os.path.isdir(INPUT_ROOT), f"Input dataset not found at {INPUT_ROOT} -- add it via '+ Add Input' first"
sys.path.insert(0, INPUT_ROOT)

os.environ['TASKB_OUT'] = '/kaggle/working/taskB_results'
os.makedirs(os.environ['TASKB_OUT'], exist_ok=True)

try:
    from dm_control import suite  # noqa
except ImportError:
    !pip install -q dm_control
    from dm_control import suite  # noqa

print('dm_control OK')

In [ ]:
import importlib
import taskB_program as P
importlib.reload(P)
print('SEEDS =', P.SEEDS)
print('CONDITIONS =', P.CONDITIONS)
print('TASKS =', list(P.TASKS.keys()))
print('WARMUP_STEPS =', P.WARMUP_STEPS, ' REFIT_EVERY =', P.REFIT_EVERY, ' REFIT_WINDOW =', P.REFIT_WINDOW)

## Optional: quick mechanical smoke test
Exercises all 3 conditions at a tiny scale (6,000 env steps, matching the local smoke test already run) in a separate output directory so it never collides with the real pilot/full-run results. Skip straight to the pilot below if you haven't touched `taskB_program.py`.

In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    os.environ['TASKB_OUT'] = '/kaggle/working/taskB_smoke'
    importlib.reload(P)
    P.set_scale(WARMUP_STEPS=1000, REFIT_EVERY=1000, REFIT_WINDOW=2000)
    P.OUT_DIR = os.environ['TASKB_OUT']
    smoke_results = P.run_all(tasks=['cartpole'], seeds=[3101], total_env_steps=6000)
    for k, r in smoke_results.items():
        print(k, '-> resid_final=', r['constraint_resid_final'], 'n_refits=', r['n_refits'])
    os.environ['TASKB_OUT'] = '/kaggle/working/taskB_results'
    importlib.reload(P)
    print('\nSmoke test passed mechanically -- ready for the pilot below.')

## Pilot (required before the full run, §3.6)
Cartpole, seed 3101, all three conditions, full 100,000 env steps. Check before proceeding to the full run:
1. **Constraint effectiveness:** in B, `|v_t^T h_tilde_t|` is ~0 throughout training after warmup (see `constraint_resid_final` below -- computed on steps strictly after the last effective refit).
2. **Training stability:** loss curves for all three conditions are finite and comparable in shape; no divergence.
3. **Reproducibility:** rerun the first 15,000 steps of condition B and confirm byte-identical results (separate cell below).
4. **Refit behavior:** consecutive `v_t` cosines are reported (`v_history.json` per cell) -- very low cosines are a finding to record, not something to tune away.

In [ ]:
pilot_results = P.run_all(tasks=['cartpole'], seeds=[3101], total_env_steps=100_000)
for k, r in pilot_results.items():
    print(k, '-> final_loss=', round(r['final_loss'], 4) if r['final_loss'] else None,
          ' resid_final=', r['constraint_resid_final'], ' n_refits=', r['n_refits'],
          ' elapsed=', round(r['elapsed_min'], 1), 'min')

In [ ]:
# Reproducibility check (pilot §3.6.3): rerun the first 15,000 steps of
# condition B TWICE, in a scratch dir, and confirm byte-identical h/kl/recon logs.
repro_dir_a = '/kaggle/working/taskB_repro_a'
repro_dir_b = '/kaggle/working/taskB_repro_b'
import shutil
shutil.rmtree(repro_dir_a, ignore_errors=True)
shutil.rmtree(repro_dir_b, ignore_errors=True)

cfg = P.XS_CONFIG.copy()
res_a = P.train_condition('cartpole', P.TASKS['cartpole'], cfg, 'B_vconstrained', 3101, repro_dir_a, total_env_steps=15000)
res_b = P.train_condition('cartpole', P.TASKS['cartpole'], cfg, 'B_vconstrained', 3101, repro_dir_b, total_env_steps=15000)

import numpy as np
sa = dict(np.load(os.path.join(repro_dir_a, 'states.npz')))
sb = dict(np.load(os.path.join(repro_dir_b, 'states.npz')))
for key in ['h', 'kl', 'recon']:
    identical = np.array_equal(sa[key], sb[key])
    print(f'{key}: byte-identical = {identical}')

## Full run (only if the pilot passes)
3 tasks x 3 conditions x 3 seeds = 27 runs, 100,000 env steps each. Declared fallback: if compute isn't available for all 27, set `tasks=['cartpole']` below for the 9-run fallback -- decide this BEFORE running, not based on partial results.

In [ ]:
FULL_TASKS = ['cartpole', 'reacher', 'pendulum']  # set to ['cartpole'] for the declared fallback
full_results = P.run_all(tasks=FULL_TASKS, total_env_steps=100_000)
print(f'\n{len(full_results)} cells complete.')

In [ ]:
import json
with open('/kaggle/working/taskB_final_manifest.json', 'w') as f:
    json.dump(full_results, f, indent=2, default=float)
print('Wrote /kaggle/working/taskB_final_manifest.json -- download this plus taskB_results/ '
      '(checkpoints, states, loss/v histories) to bring the results back into the main project '
      'for run_taskB_reemergence_analysis.py.')